In [43]:
import json
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import operator
from statistics import mean
from collections import Counter

In [44]:
with open('data/train-claims.json', 'r') as input_file:
    train_claim_data = json.load(input_file)

# Read in development data (claim)
with open('data/dev-claims.json', 'r') as input_file:
    dev_claim_data = json.load(input_file)

# Read in test data (claim)
with open('data/test-claims-unlabelled.json', 'r') as input_file:
    test_claim_data = json.load(input_file)

# Read in evidence data
with open('data/evidence.json', 'r') as input_file:
    evi_data = json.load(input_file)

train_claim_id = list(train_claim_data.keys())
train_claim_text  = [ v["claim_text"] for v in train_claim_data.values()]
dev_claim_text  = [ v["claim_text"] for v in dev_claim_data.values()]


In [45]:
import json
import nltk
from nltk.corpus import stopwords
import re

# 下载 NLTK 数据
nltk.download('punkt')

# 从文件中读取原始的evidence数据
with open('data/evidence.json', 'r') as input_file:
    evi_data = json.load(input_file)

# 预处理去除非英语证据
english_evidence = {}  # 用于存储经过预处理后的英语证据

for evi_id, evi_text in evi_data.items():
    # 判断文本是否为英语
    words = nltk.word_tokenize(evi_text)
    english_words = [word for word in words if word.isalpha()]
    if len(english_words) / len(words) > 0.5:  # 如果超过一半的词是英文单词，则认为是英语文本
        # 如果是英语，则进行进一步的预处理，例如去除停用词等
        english_text = ' '.join(english_words)
        english_evidence[evi_id] = english_text

# 输出筛选后的evidence数量
filtered_evidence_count = len(english_evidence)
print("Filtered Evidence Count:", filtered_evidence_count)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\XZH\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Filtered Evidence Count: 1181638


In [46]:
import json
import nltk
from nltk.corpus import stopwords
import re



# 提取训练集和开发集中出现频率最高的名词或者字母与数字的组合
train_claim_text = [claim_value['claim_text'] for claim_value in train_claim_data.values()]
dev_claim_text = [claim_value['claim_text'] for claim_value in dev_claim_data.values()]

# 定义停用词集合
stop_words = set(stopwords.words('english'))
# 提取名词或者字母与数字的组合
claim_words = []
for text_list in [train_claim_text, dev_claim_text]:
    for text in text_list:
        tokens = nltk.word_tokenize(text)
        for token in tokens:
            # 只保留名词或者字母与数字的组合，并且不在停用词集合中的单词
            if re.match('^[a-zA-Z0-9]+$', token) and token.lower() not in stop_words and nltk.pos_tag([token])[0][1] in ['NN', 'NNS', 'NNP', 'NNPS']:
                claim_words.append(token.lower())

# 统计出现频率最高的单词
top_words_counter = Counter(claim_words)
# 获取出现频率最高的单词列表，最多添加500个单词
top_words = [word for word, _ in top_words_counter.most_common(100)]

# 打印出现频率最高的单词列表
print("Top Words:", top_words)

# 创建一个新的 dictionary 用于存储符合条件的 evidence
climate_dic = {}

# 遍历英语证据，将包含出现频率最高的单词的 evidence 收集到 climate_dic 中
for evi_id, evi_text in english_evidence.items():
    words = nltk.word_tokenize(evi_text)
    if any(word.lower() in top_words for word in words):
        climate_dic[evi_id] = evi_text


# 训练集中的所有 evidence 加入到 climate_dic 中
for claim_info in train_claim_data.values():
    if "evidences" in claim_info:  # 检查键是否存在
        for evidence_id in claim_info["evidences"]:
            if evidence_id in evi_data:
                evi_text = evi_data[evidence_id]
                climate_dic[evidence_id] = evi_text
            else:
                print("Evidence ID not found:", evidence_id)

# 输出 climate_dic 中的 evidence 个数
print("Number of evidence in climate_dic:", len(climate_dic))

Top Words: ['climate', 'co2', 'ice', 'change', 'temperature', 'sea', 'carbon', 'years', 'temperatures', 'scientists', 'emissions', 'rise', 'earth', 'level', 'dioxide', 'greenhouse', 'record', 'past', 'ocean', 'year', 'levels', 'ipcc', 'century', 'data', 'planet', 'evidence', 'world', 'increase', 'energy', 'trend', 'heat', 'surface', 'effect', 'models', 'water', 'solar', 'human', 'gas', 'humans', 'weather', 'show', 'time', 'degrees', 'warmer', 'cause', 'changes', 'extreme', 'decades', 'found', 'greenland', 'events', 'antarctica', 'today', 'times', 'amount', 'period', 'sun', 'gases', 'polar', 'percent', 'study', 'cent', 'satellite', 'measurements', 'report', 'research', 'impact', 'oceans', 'half', 'united', 'rate', 'shows', 'states', 'cold', 'science', 'activity', 'fact', 'air', 'mean', 'summer', 'studies', 'warm', 'fossil', 'cycle', 'glaciers', 'stations', 'increases', 'age', 'australia', 'el', 'land', 'means', 'forests', 'term', 'trends', 'decline', 'celsius', 'scientist', 'power', 'li

In [47]:
# 提取测试集和开发集中所有的 evidence 的 id
train_dev_evidence_ids = set()
for claim_data in [train_claim_data, dev_claim_data]:
    for claim_info in claim_data.values():
        if "evidences" in claim_info:  # 检查键是否存在
            train_dev_evidence_ids.update(claim_info["evidences"])

# 检查测试集和开发集中的 evidence 是否都在 climate_dic 中出现
missing_evidence_ids = [evi_id for evi_id in train_dev_evidence_ids if evi_id not in climate_dic]

# 打印没有出现在 climate_dic 中的 evidence 的 id 及其数量
print("Number of evidence not in climate_dic:", len(missing_evidence_ids))
print("Missing evidence IDs:", missing_evidence_ids)


# 打印没有出现在 climate_dic 中的 evidence 的 id 对应的 evidence 内容
for missing_id in missing_evidence_ids:
    print("Evidence ID:", missing_id)
    print("Evidence Content:", evi_data[missing_id])
    print()


Number of evidence not in climate_dic: 55
Missing evidence IDs: ['evidence-617690', 'evidence-776430', 'evidence-1091064', 'evidence-946262', 'evidence-472543', 'evidence-459314', 'evidence-422399', 'evidence-181656', 'evidence-641043', 'evidence-520355', 'evidence-977782', 'evidence-380361', 'evidence-474026', 'evidence-1014366', 'evidence-1087633', 'evidence-1079220', 'evidence-179080', 'evidence-1122962', 'evidence-335697', 'evidence-702226', 'evidence-1124018', 'evidence-416776', 'evidence-1023411', 'evidence-612818', 'evidence-69294', 'evidence-499929', 'evidence-426684', 'evidence-255653', 'evidence-399707', 'evidence-971592', 'evidence-541555', 'evidence-51845', 'evidence-784155', 'evidence-1205623', 'evidence-105411', 'evidence-740261', 'evidence-956029', 'evidence-627910', 'evidence-1193739', 'evidence-995813', 'evidence-1112531', 'evidence-846906', 'evidence-901245', 'evidence-163344', 'evidence-1150262', 'evidence-462075', 'evidence-1113929', 'evidence-137771', 'evidence-787

In [48]:
# 计算测试集和开发集中所有的 evidence 的 id 的数量
total_evidence_ids = set()

for claim_data in [train_claim_data, dev_claim_data]:
    for claim_info in claim_data.values():
        if "evidences" in claim_info:  # 检查键是否存在
            total_evidence_ids.update(claim_info["evidences"])

# 打印测试集和开发集中所有的 evidence 的 id 的数量
print("Total number of evidence IDs:", len(total_evidence_ids))
# 打印测试集和开发集中所有的 evidence 的 id
print("All evidence IDs:", total_evidence_ids)


Total number of evidence IDs: 3443
All evidence IDs: {'evidence-534165', 'evidence-1124041', 'evidence-510034', 'evidence-206590', 'evidence-1194199', 'evidence-650730', 'evidence-951865', 'evidence-1078887', 'evidence-647077', 'evidence-627772', 'evidence-397800', 'evidence-1003706', 'evidence-1159946', 'evidence-908679', 'evidence-330798', 'evidence-1017374', 'evidence-161973', 'evidence-386828', 'evidence-461572', 'evidence-382966', 'evidence-335083', 'evidence-1134268', 'evidence-126736', 'evidence-690679', 'evidence-1106814', 'evidence-1120350', 'evidence-537042', 'evidence-736392', 'evidence-1133069', 'evidence-987088', 'evidence-802224', 'evidence-915493', 'evidence-226174', 'evidence-713990', 'evidence-5928', 'evidence-875615', 'evidence-959887', 'evidence-77144', 'evidence-512615', 'evidence-354463', 'evidence-246671', 'evidence-1026521', 'evidence-75512', 'evidence-843198', 'evidence-721399', 'evidence-166469', 'evidence-491841', 'evidence-371222', 'evidence-688444', 'evidenc

In [52]:
import json

# 定义要保存的文件路径
output_file_path = "climate_dic.json"

# 将 climate_dic 存储到文件中
with open(output_file_path, 'w') as output_file:
    json.dump(climate_dic, output_file)

print("Climate dictionary saved to", output_file_path)


Climate dictionary saved to climate_dic.json
